<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Research_Paper_Mining_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Assistant: arXiv & Semantic Scholar Integrator
This notebook downloads paper data, generates insights using AI, and stores everything in a searchable DuckDB database with semantic search capabilities.

In [1]:
!pip install duckdb sentence-transformers pandas arxiv scholarly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.4 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=084294f03e6f69208ec660200c98e55a98e6f9b46f5cf1085fa9463314587c2f
  Stored in directory: /root/.cache/pip/wheels/54/f8/e6/ecfceb6af875ddc5096bb3811795ac336f50371009a601454d
Successfully built bibtexparser
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib

In [2]:
import duckdb
import pandas as pd

# Initialize DuckDB
con = duckdb.connect('research_papers.db')

# Create tables for papers and insights
con.execute("""
CREATE TABLE IF NOT EXISTS papers (
    id VARCHAR PRIMARY KEY,
    source VARCHAR,
    title VARCHAR,
    authors VARCHAR,
    abstract TEXT,
    categories VARCHAR,
    citations INTEGER,
    summary TEXT,
    key_insights TEXT,
    future_trends TEXT,
    embedding FLOAT[]
)
""")
print("Database initialized.")

Database initialized.


In [1]:
import arxiv
from scholarly import scholarly
from sentence_transformers import SentenceTransformer

# Initialize Embedding Model
model = SentenceTransformer('all-MiniLM-L6-v2')

def fetch_arxiv_papers(query, max_results=5):
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)

    papers = []
    for result in client.results(search):
        papers.append({
            'id': result.entry_id,
            'source': 'arXiv',
            'title': result.title,
            'authors': ', '.join(a.name for a in result.authors),
            'abstract': result.summary,
            'categories': ', '.join(result.categories),
            'citations': 0  # arXiv doesn't provide citations directly via this API
        })
    return papers

def fetch_semantic_scholar_papers(query, limit=5):
    # Using scholarly as a proxy for Semantic Scholar/Google Scholar data
    search_query = scholarly.search_pubs(query)
    papers = []
    for i in range(limit):
        try:
            result = next(search_query)
            bib = result.get('bib', {})
            papers.append({
                'id': f"scholar_{i}_{hash(bib.get('title', ''))}",
                'source': 'Semantic Scholar',
                'title': bib.get('title', 'N/A'),
                'authors': bib.get('author', 'N/A'),
                'abstract': bib.get('abstract', 'No abstract available'),
                'categories': 'N/A',
                'citations': result.get('num_citations', 0)
            })
        except StopIteration:
            break
    return papers

def process_and_store(paper_list):
    for p in paper_list:
        # Placeholder for AI generation logic
        summary = f"Summary of {p['title']}..."
        key_insights = "1. Insight A\n2. Insight B"
        future_trends = "Expansion in X and Y."

        # Generate embedding from title + abstract
        text_to_embed = f"{p['title']} {p['abstract']}"
        embedding = model.encode(text_to_embed).tolist()

        con.execute("""
            INSERT OR IGNORE INTO papers
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (p['id'], p['source'], p['title'], p['authors'], p['abstract'],
              p['categories'], p['citations'], summary, key_insights, future_trends, embedding))

# Example Run
query = "Large Language Models efficiency"
arxiv_data = fetch_arxiv_papers(query)
scholar_data = fetch_semantic_scholar_papers(query)

process_and_store(arxiv_data + scholar_data)
print(f"Processed and stored {len(arxiv_data) + len(scholar_data)} papers.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

NameError: name 'con' is not defined

In [3]:
import duckdb

# Re-ensure connection exists
con = duckdb.connect('research_papers.db')

def semantic_search(query_text, top_k=3):
    query_embedding = model.encode(query_text).tolist()

    # Use DuckDB list_cosine_similarity for searching
    results = con.execute("""
        SELECT title, authors, source, list_cosine_similarity(embedding, ?) as score
        FROM papers
        ORDER BY score DESC
        LIMIT ?
    """, (query_embedding, top_k)).fetchdf()

    return results

# Test Search
print("Search Results for 'efficient inference':")
print(semantic_search("efficient inference"))

Search Results for 'efficient inference':
Empty DataFrame
Columns: [title, authors, source, score]
Index: []


In [4]:
# Check if data actually exists in the table
row_count = con.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
print(f"Total papers in database: {row_count}")

if row_count > 0:
    print("\nFirst 2 entries:")
    print(con.execute("SELECT title, source FROM papers LIMIT 2").fetchdf())
else:
    print("\nDatabase is empty. Re-running the storage process...")
    process_and_store(arxiv_data + scholar_data)
    new_count = con.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
    print(f"New paper count: {new_count}")

Total papers in database: 0

Database is empty. Re-running the storage process...
New paper count: 10
